# 03 Feature Engineering

This notebook is part of the end-to-end bank loan propensity prediction and MLOps deployment project.

## Importing Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split

## Loading Data

In [2]:
# Load Cleaned Customer Dataset

# Define data directory
INTERIM_DIR = Path("../data/processed")

# Load cleaned customer dataset
cust_data = pd.read_csv(
    INTERIM_DIR / "02_cleaned_customer_data.csv"
)

cust_data.head()

,ID,Age,CustomerSince,HighestSpend,ZipCode,HiddenScore,MonthlyAverageSpend,Level,Mortgage,Security,FixedDepositAccount,InternetBanking,CreditCard,LoanOnCard
0,10,34,9,180,93023,1,8.9,3,0,0,0,0,0,1
1,11,65,39,105,94710,4,2.4,3,0,0,0,0,0,0
2,12,29,5,45,90277,3,0.1,2,0,0,0,1,0,0
3,13,48,23,114,93106,2,3.8,3,0,1,0,0,0,0
4,14,59,32,40,94920,4,2.5,2,0,0,0,1,0,0


## Data Validation

In [3]:
# Final Data Validation Before Modeling

print("Dataset Shape:", cust_data.shape)

dtype_summary = pd.DataFrame({
    "Feature": cust_data.columns,
    "Data Type": cust_data.dtypes.values,
    "Non-Null Count": cust_data.count().values,
    "Missing Values": cust_data.isna().sum().values,
    "Duplicate Values": cust_data.isna().sum().values
})

dtype_summary

Dataset Shape: (4980, 14)


,Feature,Data Type,Non-Null Count,Missing Values,Duplicate Values
0,ID,int64,4980,0,0
1,Age,int64,4980,0,0
2,CustomerSince,int64,4980,0,0
3,HighestSpend,int64,4980,0,0
4,ZipCode,int64,4980,0,0
5,HiddenScore,int64,4980,0,0
6,MonthlyAverageSpend,float64,4980,0,0
7,Level,int64,4980,0,0
8,Mortgage,int64,4980,0,0
9,Security,int64,4980,0,0


## Target Variable Preparation

In [4]:
# Target Variable Preparation

# Convert target variable from float to integer after removing missing target values
cust_data["LoanOnCard"] = cust_data["LoanOnCard"].astype(int)

print("Target Data Type:", cust_data["LoanOnCard"].dtype)

# Target Variable Distribution

target_distribution = pd.DataFrame({
    "Count": cust_data["LoanOnCard"].value_counts(),
    "Percentage (%)": (
        cust_data["LoanOnCard"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
})

target_distribution.index.name = "LoanOnCard"

target_distribution

Target Data Type: int32


,Count,Percentage (%)
LoanOnCard,,
0,4500,90.36
1,480,9.64


## Recreate Feature Groups

In [5]:
# Define Feature Groups

target_col = "LoanOnCard"

identifier_col = "ID"

high_cardinality_col = "ZipCode"

numerical_cols = [
    "Age",
    "CustomerSince",
    "HighestSpend",
    "MonthlyAverageSpend",
    "Mortgage"
]

ordinal_cols = [
    "Level",
    "HiddenScore"
]

binary_cols = [
    "Security",
    "FixedDepositAccount",
    "InternetBanking",
    "CreditCard"
]

## Remove Non-Predictive Identifier

In [6]:
# Remove Non-Predictive Identifier

# ID uniquely identifies each customer and does not provide predictive value
cust_data = cust_data.drop(columns=[identifier_col])

print("Dataset Shape After Dropping ID:", cust_data.shape)

Dataset Shape After Dropping ID: (4980, 13)


## Evaluate and Remove ZipCode

In [7]:
# Evaluate High Cardinality Geographic Feature

zipcode_unique_count = cust_data[high_cardinality_col].nunique()

print("Number of Unique ZipCode Values:", zipcode_unique_count)
print("Total Records:", cust_data.shape[0])
print("ZipCode Cardinality Ratio:", round(zipcode_unique_count / cust_data.shape[0], 4))

Number of Unique ZipCode Values: 467
Total Records: 4980
ZipCode Cardinality Ratio: 0.0938


In [8]:
# Remove High Cardinality ZipCode Feature

# ZipCode has high cardinality and no external geographic mapping is available.
# Keeping it may create sparse features and increase model complexity.
cust_data = cust_data.drop(columns=[high_cardinality_col])

print("Dataset Shape After Dropping ZipCode:", cust_data.shape)

Dataset Shape After Dropping ZipCode: (4980, 12)


# Test before creating the feature

## Mortgage Feature Review

In [9]:
# Mortgage Zero vs Non-Zero Review

mortgage_flag_temp = (cust_data["Mortgage"] > 0).astype(int)

mortgage_flag_summary = pd.crosstab(
    mortgage_flag_temp,
    cust_data[target_col],
    normalize="index"
).mul(100).round(2)

mortgage_flag_summary.index = ["No Mortgage", "Has Mortgage"]

display(mortgage_flag_summary)

LoanOnCard,0,1
No Mortgage,90.95,9.05
Has Mortgage,89.04,10.96


### Observation

Loan Rate Increase
= 10.96% - 9.05%
= 1.91%

That's a very small separation.

The "amount of mortgage" appears to contain more information than simply "Mortgage > 0"

Because:

Mortgage Amount contains:
- ownership + size of mortgage

while

HasMortgage contains:
- ownership only

That refers to information loss.

In [10]:
contingency_table = pd.crosstab(
    mortgage_flag_temp,
    cust_data["LoanOnCard"]
)

chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print("P-value:", p_value)

P-value: 0.040024537805997414


## Observation

P-value = 0.0400

Since: 0.0400 < 0.05

the association is statistically significant.

However...

Why I Still Wouldn't Automatically Add HasMortgage

Look at the actual effect size:

| Mortgage Status | Loan Rate |
|-----------------|-----------|
| No Mortgage | 9.05% |
| Has Mortgage | 10.96% |

Difference: 1.91 percentage points

With 4,980 observations, very small differences can become statistically significant.

This means:

Statistically Significant ≠ Predictively Important

### Decision:

A HasMortgage feature will not be added automatically in Phase 4.

Mortgage already contains both ownership and amount information. A separate mortgage ownership flag may be tested during Phase 5 only if it provides additional predictive value.

## Create Final Modeling Feature List

In [11]:
# Create Final Modeling Feature List

model_features = (
    numerical_cols
    + ordinal_cols
    + binary_cols
)

print("Total Modeling Features:", len(model_features))
print(model_features)

Total Modeling Features: 11
['Age', 'CustomerSince', 'HighestSpend', 'MonthlyAverageSpend', 'Mortgage', 'Level', 'HiddenScore', 'Security', 'FixedDepositAccount', 'InternetBanking', 'CreditCard']


## Create Alternative Feature Sets for Phase 5

In [12]:
# Create Alternative Feature Sets for Multicollinearity Experiments

# Feature Set A: All features
features_all = model_features.copy()

# Feature Set B: Remove Age
features_no_age = [
    col for col in model_features
    if col != "Age"
]

# Feature Set C: Remove CustomerSince
features_no_customersince = [
    col for col in model_features
    if col != "CustomerSince"
]

print("All Features:", len(features_all))
print(features_all)

print("\nNo Age:", len(features_no_age))
print(features_no_age)

print("\nNo CustomerSince:", len(features_no_customersince))
print(features_no_customersince)

All Features: 11
['Age', 'CustomerSince', 'HighestSpend', 'MonthlyAverageSpend', 'Mortgage', 'Level', 'HiddenScore', 'Security', 'FixedDepositAccount', 'InternetBanking', 'CreditCard']

No Age: 10
['CustomerSince', 'HighestSpend', 'MonthlyAverageSpend', 'Mortgage', 'Level', 'HiddenScore', 'Security', 'FixedDepositAccount', 'InternetBanking', 'CreditCard']

No CustomerSince: 10
['Age', 'HighestSpend', 'MonthlyAverageSpend', 'Mortgage', 'Level', 'HiddenScore', 'Security', 'FixedDepositAccount', 'InternetBanking', 'CreditCard']


## Separate Features and Target

In [13]:
# Separate Features and Target Variable

X = cust_data[features_all].copy()
y = cust_data[target_col].copy()

print("Feature Matrix Shape :", X.shape)
print("Target Vector Shape  :", y.shape)

Feature Matrix Shape : (4980, 11)
Target Vector Shape  : (4980,)


## Stratified Train-Test Split

In [14]:
# Stratified Train-Test Split

# Stratification is used because the target variable is imbalanced.
# This preserves the loan/non-loan ratio in both training and testing sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape :", y_test.shape)

X_train Shape: (3984, 11)
X_test Shape : (996, 11)
y_train Shape: (3984,)
y_test Shape : (996,)


## Verify Target Distribution After Split

In [15]:
# Verify Target Distribution After Train-Test Split

train_distribution = (
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame(name="Train Percentage")
)

test_distribution = (
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame(name="Test Percentage")
)

display(train_distribution)
display(test_distribution)

,Train Percentage
LoanOnCard,
0,90.36
1,9.64


,Test Percentage
LoanOnCard,
0,90.36
1,9.64


## Create Train-Test Versions for Alternative Feature Sets

In [16]:
# Create Alternative Training and Testing Feature Sets

# These datasets will be used in Phase 5 to evaluate the effect of
# removing Age or CustomerSince for models sensitive to multicollinearity.

X_train_all = X_train[features_all].copy()
X_test_all = X_test[features_all].copy()

X_train_no_age = X_train[features_no_age].copy()
X_test_no_age = X_test[features_no_age].copy()

X_train_no_customersince = X_train[features_no_customersince].copy()
X_test_no_customersince = X_test[features_no_customersince].copy()

print("All Features Train Shape:", X_train_all.shape)
print("No Age Train Shape:", X_train_no_age.shape)
print("No CustomerSince Train Shape:", X_train_no_customersince.shape)

All Features Train Shape: (3984, 11)
No Age Train Shape: (3984, 10)
No CustomerSince Train Shape: (3984, 10)


## Save Prepared Data for Phase 5

In [17]:
# Save Prepared Data for Model Training

# Combine X and y for saving clean train/test datasets
train_data = X_train.copy()
train_data[target_col] = y_train

test_data = X_test.copy()
test_data[target_col] = y_test

train_data.to_csv("../data/processed/03_train_data.csv", index=False)
test_data.to_csv("../data/processed/03_test_data.csv", index=False)

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


## Phase 4 Summary

In this phase, the cleaned dataset was prepared for machine learning model development.

The target variable was converted from float to integer after missing target observations had already been removed during data cleaning. The ID column was removed because it is a unique customer identifier and does not provide predictive value.

ZipCode was removed due to high cardinality and lack of external geographic context. Retaining this feature would increase dimensionality and may introduce sparse, noisy features.

Financial variables were reviewed for skewness. However, log transformations were not applied globally during this phase because different model families respond differently to skewed variables. Log-transformed alternatives will be evaluated during model training where appropriate.

Although Age and CustomerSince showed severe multicollinearity during EDA, both variables were retained in the main feature set. Alternative feature sets were created to support empirical comparison during model training:
- All features
- Features excluding Age
- Features excluding CustomerSince

A stratified train-test split was used to preserve the original class imbalance in both training and testing datasets.

The prepared datasets are now ready for Phase 5: Model Training and Evaluation.

# Phase 4 Deliverables Completed

- Final Dataset Validation
- Target Variable Preparation
- Feature Group Definition
- Identifier Feature Removal
- High Cardinality Feature Evaluation
- ZipCode Removal
- Mortgage Feature Engineering Assessment
- Feature Selection Strategy Definition
- Multicollinearity Handling Strategy
- Modeling Feature List Creation
- Alternative Feature Set Creation
- Feature and Target Separation
- Stratified Train-Test Split
- Class Distribution Validation
- Model-Specific Dataset Preparation
- Prepared Dataset Export
- Data Preparation & Feature Engineering Summary